In [2]:
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Загрузка данных
df = pd.read_csv('tower_dataset_ml.csv')

# Очистка данных: удаляем строки, где все или некоторые значения пустые (NaN)
df = df.dropna()

# 2. Подготовка признаков (X) и целевой переменной (y)
# Колонка 'tag' - идентификатор, ее убираем из обучения
X = df.drop(columns=['tag', 'weight_kg'])
y = df['weight_kg']

# 3. Разделение на обучающую и тестовую выборки (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Построение ML пайплайна
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

# 5. Обучение модели
pipeline.fit(X_train, y_train)

# 6. Валидация и оценка качества
y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== Результаты модели для Колонных аппаратов на тестовой выборке ===")
print(f"MAE (Средняя абсолютная ошибка): {mae:.2f} кг")
print(f"RMSE (Корень из среднеквадратичной ошибки): {rmse:.2f} кг")
print(f"R2 Score (Коэффициент детерминации): {r2:.4f}")

# 7. Анализ важности признаков (Feature Importance)
importances = pipeline.named_steps['model'].feature_importances_
feature_names = X.columns

print("\n=== Важность признаков ===")
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(importance_df.to_string(index=False))

filename = 'tower_weight_model.joblib'
joblib.dump(pipeline, filename)

print(f"\n[OK] Модель успешно сохранена в файл: {filename}")

=== Результаты модели для Колонных аппаратов на тестовой выборке ===
MAE (Средняя абсолютная ошибка): 642.36 кг
RMSE (Корень из среднеквадратичной ошибки): 1313.16 кг
R2 Score (Коэффициент детерминации): 0.9997

=== Важность признаков ===
  Feature  Importance
  ss_dist    0.541445
trays_num    0.359274
 diameter    0.078281
 pressure    0.021001

[OK] Модель успешно сохранена в файл: tower_weight_model.joblib
